In [107]:
import pandas as pd
import numpy as np
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [108]:
torch.manual_seed(42)

#### **Loading data**

In [109]:
processed_dir = '../data/processed'
df = pd.read_parquet(f'{processed_dir}/master_data.parquet')
movies = pd.read_parquet(f'{processed_dir}/movies_clean.parquet')

#### **Time split**

In [110]:
df = df.sort_values('datetime')

split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

#### **Data preprocessing**

In [111]:
genres_dummies = movies['genres'].str.get_dummies(sep='|')
genres_dummies.columns = [f'genre_{c}' for c in genres_dummies.columns]
movies = pd.concat([movies, genres_dummies], axis=1)
genre_cols = list(genres_dummies.columns)

if 'release_year' not in movies.columns:
    movies['release_year'] = movies['title'].str.extract(r'\((\d{4})\)').astype(float)
movies['release_year'] = movies['release_year'].fillna(movies['release_year'].median())
movies['decade'] = (movies['release_year'] // 10 * 10).astype(int)

In [112]:
movie_stats = train_df.groupby('movieId').agg(
    movie_rating_count=('rating', 'count'),
    movie_rating_mean=('rating', 'mean'),
    movie_rating_std=('rating', 'std')
).reset_index()
movie_stats['movie_rating_std'] = movie_stats['movie_rating_std'].fillna(0)

user_stats = train_df.groupby('userId').agg(
    user_rating_count=('rating', 'count'),
    user_rating_mean=('rating', 'mean'),
    user_rating_std=('rating', 'std')
).reset_index()
user_stats['user_rating_std'] = user_stats['user_rating_std'].fillna(0)

In [113]:
static_cols = ['movieId'] + genre_cols

train_df = train_df.merge(movies[static_cols], on='movieId', how='left')
test_df = test_df.merge(movies[static_cols], on='movieId', how='left')

movie_to_year = dict(zip(movies['movieId'], movies['release_year']))
movie_to_decade = dict(zip(movies['movieId'], movies['decade']))

for dataset in [train_df, test_df]:
    dataset['release_year'] = dataset['movieId'].map(movie_to_year)
    dataset['decade'] = dataset['movieId'].map(movie_to_decade)

train_df = train_df.merge(movie_stats, on='movieId', how='left')
train_df = train_df.merge(user_stats, on='userId', how='left')

test_df = test_df.merge(movie_stats, on='movieId', how='left')
test_df = test_df.merge(user_stats, on='userId', how='left')

##### Cold start handling

In [114]:
global_train_mean = train_df['rating'].mean()

for dataset in [train_df, test_df]:
    dataset['user_rating_mean'] = dataset['user_rating_mean'].fillna(global_train_mean)
    dataset['movie_rating_mean'] = dataset['movie_rating_mean'].fillna(global_train_mean)
    dataset['user_rating_count'] = dataset['user_rating_count'].fillna(0)
    dataset['user_rating_std'] = dataset['user_rating_std'].fillna(0)
    dataset['movie_rating_count'] = dataset['movie_rating_count'].fillna(0)
    dataset['movie_rating_std'] = dataset['movie_rating_std'].fillna(0)

#### **Scaling and encoding**

In [115]:
for dataset in [train_df, test_df]:
    dataset['rating_year'] = pd.to_datetime(dataset['datetime']).dt.year
    dataset['movie_age_at_rating'] = (dataset['rating_year'] - dataset['release_year']).clip(lower=0)

continuous_features = [
    'user_rating_count', 'user_rating_mean', 'user_rating_std',
    'movie_rating_count', 'movie_rating_mean', 'movie_rating_std',
    'release_year', 'movie_age_at_rating'
]

scaler = StandardScaler()
train_df[continuous_features] = scaler.fit_transform(train_df[continuous_features])
test_df[continuous_features] = scaler.transform(test_df[continuous_features])

user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
decade_encoder = LabelEncoder()

user_encoder.fit(df['userId'])
item_encoder.fit(df['movieId'])
decade_encoder.fit(movies['decade'])

train_df['user_idx'] = user_encoder.transform(train_df['userId'])
train_df['item_idx'] = item_encoder.transform(train_df['movieId'])
train_df['decade_idx'] = decade_encoder.transform(train_df['decade'])

test_df['user_idx'] = user_encoder.transform(test_df['userId'])
test_df['item_idx'] = item_encoder.transform(test_df['movieId'])
test_df['decade_idx'] = decade_encoder.transform(test_df['decade'])

num_users = len(user_encoder.classes_)
num_items = len(item_encoder.classes_)
num_decades = len(decade_encoder.classes_)

#### **Dataset class**

In [116]:
class AdvancedHybridMovieDataset(Dataset):
    def __init__(self, users, items, decades, continuous_feats, genres_multi, ratings):
        self.users = torch.tensor(users, dtype=torch.long)
        self.items = torch.tensor(items, dtype=torch.long)
        self.decades = torch.tensor(decades, dtype=torch.long)
        
        self.continuous_feats = torch.tensor(continuous_feats, dtype=torch.float32)
        self.genres_multi = torch.tensor(genres_multi, dtype=torch.float32) 
        
        self.ratings = torch.tensor(ratings, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return (
            self.users[idx], 
            self.items[idx], 
            self.decades[idx], 
            self.continuous_feats[idx], 
            self.genres_multi[idx], 
            self.ratings[idx]
        )

train_loader = DataLoader(
    AdvancedHybridMovieDataset(
        train_df['user_idx'].values, 
        train_df['item_idx'].values, 
        train_df['decade_idx'].values,
        train_df[continuous_features].values, 
        train_df[genre_cols].values,          
        train_df['rating'].values
    ), 
    batch_size=1024, 
    shuffle=True,
    drop_last=True
)

test_loader = DataLoader(
    AdvancedHybridMovieDataset(
        test_df['user_idx'].values, 
        test_df['item_idx'].values, 
        test_df['decade_idx'].values,
        test_df[continuous_features].values, 
        test_df[genre_cols].values, 
        test_df['rating'].values
    ), 
    batch_size=1024, 
    shuffle=False,
    drop_last=True
)

#### **Model immplementation**

In [125]:
class AdvancedHybridTwoTowerModel(nn.Module):
    def __init__(self, num_users, num_items, num_decades, num_continuous_feats, num_genre_cols, embed_dim=32, dropout_rate=0.3, global_mean=3.5):
        super().__init__()
        
        self.global_mean = global_mean
        
        self.user_embed = nn.Embedding(num_users, embed_dim)
        
        self.user_dense_process = nn.Sequential(
            nn.Linear(3, 8),
            nn.GELU()
        )
        
        self.user_tower = nn.Sequential(
            nn.Linear(embed_dim + 8, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 32)
        )
        
        self.item_embed = nn.Embedding(num_items, embed_dim)
        self.decade_embed = nn.Embedding(num_decades, embed_dim)
        
        self.genre_process = nn.Sequential(
            nn.Linear(num_genre_cols, embed_dim // 2),
            nn.GELU()
        )
        
        self.item_dense_process = nn.Sequential(
            nn.Linear(5, 16),
            nn.GELU()
        )
        
        self.item_tower = nn.Sequential(
            nn.Linear(embed_dim * 2 + (embed_dim // 2) + 16, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 32)
        )
        
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        
        self.prediction_layer = nn.Sequential(
            nn.Linear(32 + 32, 32),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, 1)
        )
        
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_idx, item_idx, decade_idx, continuous_feats, genres_multi):
        u_emb = self.user_embed(user_idx)
        u_dense = self.user_dense_process(continuous_feats[:, :3])
        u_combined = torch.cat([u_emb, u_dense], dim=1)
        u_vector = self.user_tower(u_combined)
        
        i_emb = self.item_embed(item_idx)
        d_emb = self.decade_embed(decade_idx)
        g_emb = self.genre_process(genres_multi)
        i_dense = self.item_dense_process(continuous_feats[:, 3:])
        
        i_combined = torch.cat([i_emb, d_emb, g_emb, i_dense], dim=1)
        i_vector = self.item_tower(i_combined)
        
        combined = torch.cat([u_vector, i_vector], dim=1)
        out = self.prediction_layer(combined)
        
        u_b = self.user_bias(user_idx)
        i_b = self.item_bias(item_idx)
        
        final_out = self.global_mean + u_b + i_b + out
        return final_out.squeeze()

In [126]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AdvancedHybridTwoTowerModel(
    num_users=num_users,
    num_items=num_items,
    num_decades=num_decades,
    num_continuous_feats=len(continuous_features), 
    num_genre_cols=len(genre_cols),              
    embed_dim=64, 
    global_mean=train_df['rating'].mean()
).to(device)

criterion = nn.SmoothL1Loss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,              
    weight_decay=1e-4     
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=1
)

#### **Training loop**

In [127]:
epochs = 8

best_val_loss = float('inf')
best_model_state = None

print(f"Rozpoczynanie treningu rozszerzonego modelu na {device}...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    start_time = time.time()
    
    for users, items, decades, continuous_feats, genres_multi, ratings in train_loader:
        users = users.to(device)
        items = items.to(device)
        decades = decades.to(device)
        continuous_feats = continuous_feats.to(device)
        genres_multi = genres_multi.to(device)
        ratings = ratings.to(device)
        
        optimizer.zero_grad()

        predictions = model(users, items, decades, continuous_feats, genres_multi)
        loss = criterion(predictions, ratings)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        total_loss += loss.item()
        
    avg_train_loss = total_loss / len(train_loader)

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for users, items, decades, continuous_feats, genres_multi, ratings in test_loader:  
            users = users.to(device)
            items = items.to(device)
            decades = decades.to(device)
            continuous_feats = continuous_feats.to(device)
            genres_multi = genres_multi.to(device)
            ratings = ratings.to(device)

            preds = model(users, items, decades, continuous_feats, genres_multi)
            loss = criterion(preds, ratings)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(test_loader)

    scheduler.step(avg_val_loss)

    epoch_time = time.time() - start_time

    print(
        f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train: {avg_train_loss:.4f} | "
        f"Val: {avg_val_loss:.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.6f} | "
        f"Czas: {epoch_time:.2f}s"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = model.state_dict().copy() 

if best_model_state is not None:
    model.load_state_dict(best_model_state)

Rozpoczynanie treningu rozszerzonego modelu na cpu...
Epoch 01/8 | Train: 0.3394 | Val: 0.4404 | LR: 0.000300 | Czas: 6.92s
Epoch 02/8 | Train: 0.2872 | Val: 0.4321 | LR: 0.000300 | Czas: 7.22s
Epoch 03/8 | Train: 0.2817 | Val: 0.4319 | LR: 0.000300 | Czas: 7.24s
Epoch 04/8 | Train: 0.2774 | Val: 0.4353 | LR: 0.000300 | Czas: 8.31s
Epoch 05/8 | Train: 0.2746 | Val: 0.4355 | LR: 0.000150 | Czas: 13.64s
Epoch 06/8 | Train: 0.2712 | Val: 0.4365 | LR: 0.000150 | Czas: 11.77s
Epoch 07/8 | Train: 0.2699 | Val: 0.4378 | LR: 0.000075 | Czas: 10.53s
Epoch 08/8 | Train: 0.2672 | Val: 0.4377 | LR: 0.000075 | Czas: 14.46s


#### **Evaluation**

In [128]:
model.eval()
test_predictions, test_actuals = [], []

with torch.no_grad():
    for users, items, decades, continuous_feats, genres_multi, ratings in test_loader:
        users = users.to(device)
        items = items.to(device)
        decades = decades.to(device)
        continuous_feats = continuous_feats.to(device)
        genres_multi = genres_multi.to(device)

        preds = model(users, items, decades, continuous_feats, genres_multi)

        preds = torch.clamp(preds, 0.5, 5.0)

        test_predictions.extend(preds.cpu().numpy())
        
        test_actuals.extend(ratings.cpu().numpy())

rmse = np.sqrt(mean_squared_error(test_actuals, test_predictions))
print(f"\nEnhanced Hybrid Two-Tower RMSE: {rmse:.4f}")


Enhanced Hybrid Two-Tower RMSE: 1.0327
